# Voice under Stress - Analysis

import of all functions needed

In [ ]:
import os
import sys
import pandas as pd
import datetime as dt
import imageio
import numpy as np
import subprocess
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
import scipy 
import math
import shap

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, precision_recall_curve, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, LeaveOneOut, StratifiedKFold, StratifiedShuffleSplit, train_test_split
from sklearn import metrics, svm, linear_model
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

from scipy import stats
from scipy.stats import skew, kurtosis
from statsmodels.stats.descriptivestats import sign_test

# fix random seed for reproducibility
np.random.seed(42)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
homepath=os.getcwd()
scripts_path=os.path.join(homepath, 'scripts')
sys.path.append(scripts_path)
import myml
import mystress
import myvoice

In [ ]:
# Check version of Python==64
!python -c "import sys; print(sys.maxsize > 2**32)"

# random seed for reproducability
np.random.seed(42)

# show all columns of dataframes
pd.set_option('display.max_columns', None)

# set current dir to highest hierachy to add data path
os.chdir('/')
#data_folder='/data'
data_folder='/data'
sys.path.append(data_folder)
os.chdir(homepath)

# Main Code

### Step 1: load data and calculate variables

In [ ]:
filename_librosa='./processed_nemo_data/features.csv'
filename_behavior=data_folder + '/raw/TSST_behavior.csv'
filename_praat='./processed_nemo_data/new_praat_results'
filename_opensmile='./processed_nemo_data/opensmile_features.csv'
filename_output='./processed_nemo_data/df_vpn'

df, librosa_featurenames, praat_featurenames, opensimle_feature=mystress.load_df(filename_opensmile, filename_praat, 
                                                 filename_librosa, filename_behavior, filename_output)
df=mystress.calc_var(df)

In [ ]:
audio=praat_featurenames +librosa_featurenames[0:40]+list(opensimle_feature)
audio.append('Sex')

In [ ]:
small_audio=['meanF0Hz', 'stdevF0Hz', 'HNR', 'localJitter', 'localShimmer', 'energy', 'length', 'Sex']

# Classification

In [ ]:
X, y, vpn = myml.split_and_shuffle(df,  audio, 'Cond')

# Simple Logistic Regression

### without any parameter tuning.

In [ ]:
m = LogisticRegression(
    C=0.5,
    solver="saga",
    l1_ratio=0,      # L2
    max_iter=10000
)

res_lr = myml.classification_10foldcv(X, y, m, 'LR', 'yes')
res_lr.columns = ["y_true", "y_score"] 


### with hyperparameter tuning (nested cv)

Here we are using a 10-fold Stratified Cross-Validation with nested 3-fold CV for hyperparameter tuning.
In each outer fold, the best model is selected using GridSearchCV on the training data
and then evaluated on the held-out test fold. Performance is compared to a majority
class baseline and repeated for all folds.



In [ ]:
m = LogisticRegression(
    solver="saga",
    max_iter=10000,
    class_weight="balanced"
)
space = {
    "C": [10, 1, 0.5, 0.1, 0.01],
    "l1_ratio": [0, 1]   # L2 vs L1
}

feat_imp, importance_df, res_lr_nested = myml.classification_10foldcv_nested_feature_importance_LR(X, y, m, space, 'yes', audio=audio, df=df)
res_lr_nested.columns = ["y_true", "y_score"] 

### with hyperparameter tuning (nested cv) and pca

Here we are using a 10-fold Stratified Cross-Validation with nested 3-fold CV for hyperparameter tuning.
A pipeline consisting of StandardScaler, PCA, and the classifier is used.
In each outer fold, the training data is used in a GridSearchCV with 3-fold CV
to select the best hyperparameters. PCA and scaling are fitted only on the
training data within each fold to avoid data leakage. The best pipeline is
then evaluated on the held-out test fold. This process is repeated for all folds.

In [ ]:
m = LogisticRegression(
    solver="saga",
    max_iter=10000,
    class_weight="balanced"
)
space = {
    "model__C": [10, 1, 0.5, 0.1, 0.01],
    "model__l1_ratio": [0, 1],   # L2 vs L1
    "pca__n_components": [5, 10, 15, 20, 25, 30]
}

res_lr_nested_pca = myml.classification_10foldcv_nested_with_pca(X, y, m, space, 'yes')
res_lr_nested_pca.columns = ["y_true", "y_score"] 

# Random Forest Classifier

#### Without Hyperparameter-Tuning

In [ ]:
model = RandomForestClassifier(n_estimators=1000, max_depth=2, max_features='sqrt', min_samples_leaf=3)

res_rf = myml.classification_10foldcv(X, y, model, 'RF')
res_rf.columns = ["y_true", "y_score"] 

### with hyperparameter tuning (nested cv)

Here we are using a 10-fold Stratified Cross-Validation with nested 3-fold CV for hyperparameter tuning.
In each outer fold, the best model is selected using GridSearchCV on the training data
and then evaluated on the held-out test fold. Performance is compared to a majority
class baseline and repeated for all folds.

In [ ]:
model = RandomForestClassifier(random_state=1, n_estimators=1000)

space = dict()
space['max_depth'] = [1, 2, 4, 8] #, 16, 32, 64]
space ['min_samples_leaf']=[1, 2, 4] #, 8, 16, 32]

res_rf_nested = myml.classification_10foldcv_nested(X, y, model, 'rf - nested', space, 'no')
res_rf_nested.columns = ["y_true", "y_score"] 

## Support Vector Machine

### without any hyperparameter tuning

In [ ]:
model = SVC(kernel='linear', degree=3, gamma='scale', probability=True, class_weight='balanced')

res_svm = myml.classification_10foldcv(X, y, model, 'SVM', 'yes')
res_svm.columns = ["y_true", "y_score"] 

### with hyperparameter tuning (nested cv)

Here we are using a 10-fold Stratified Cross-Validation with nested 3-fold CV for hyperparameter tuning.
In each outer fold, the best model is selected using GridSearchCV on the training data
and then evaluated on the held-out test fold. Performance is compared to a majority
class baseline and repeated for all folds.

In [ ]:
model = SVC(kernel='rbf', degree=3, gamma='scale', probability=True, class_weight='balanced')

space = dict()
space['C'] = [10, 1.0, 0.1, 0.01]
space['kernel'] = ['linear', 'rbf']
gamma = ['scale',0.001, 0.01, 0.1, 1, 10]
 
res_svm_nested = myml.classification_10foldcv_nested(X, y, model, 'svm - nested', space, 'yes')
res_svm_nested.columns = ["y_true", "y_score"] 


### with hyperparameter tuning (nested cv) and pca

Here we are using a 10-fold Stratified Cross-Validation with nested 3-fold CV for hyperparameter tuning.
A pipeline consisting of StandardScaler, PCA, and the classifier is used.
In each outer fold, the training data is used in a GridSearchCV with 3-fold CV
to select the best hyperparameters. PCA and scaling are fitted only on the
training data within each fold to avoid data leakage. The best pipeline is
then evaluated on the held-out test fold. This process is repeated for all folds.

In [ ]:
model = SVC(kernel='rbf', degree=3, gamma='scale', probability=True, class_weight='balanced')

space = {
    "pca__n_components": [5, 10, 15, 20, 25, 30],
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ['linear', 'rbf'],
    "gamma" : ['scale', 0.001, 0.01, 0.1, 1]
}

res_svm_nested_pca = myml.classification_10foldcv_nested_with_pca(X, y, model, space, 'yes')
res_svm_nested_pca.columns = ["y_true", "y_score"] 

## XGBoost

Here we are using a 10-fold Stratified Cross-Validation for model evaluation.
In each fold, the model is trained on the training set and evaluated on the
held-out test fold. Performance metrics (Accuracy, F1) are aggregated across
folds and overall predictions are used to compute ROC and confusion matrix.

In [ ]:
model = XGBClassifier(
    max_depth=1,
    gamma=2,
    eta=0.7,
    reg_alpha=0.7,
    reg_lambda=0.7,
    eval_metric='logloss'
)

res_xgb = myml.classification_10foldcv(X, y, model, 'xgb', 'yes')
res_xgb.columns = ["y_true", "y_score"] 

## XGBoost - nested

### with hyperparameter tuning (nested cv)

Here we are using a 10-fold Stratified Cross-Validation with nested 3-fold CV for hyperparameter tuning.
In each outer fold, the best model is selected using GridSearchCV on the training data
and then evaluated on the held-out test fold. Performance is compared to a majority
class baseline and repeated for all folds.

In [ ]:
model = XGBClassifier(
    max_depth=1,
    gamma=2,
    eta=0.7,
    reg_alpha=0.7,
    reg_lambda=0.7,
    eval_metric='logloss'
)

space = {
    "max_depth": [1, 2, 4, 8],
    "learning_rate": [0.03, 0.1, 0.2],
    "n_estimators": [50, 100, 150],
}

res_xgb_nested = myml.classification_10foldcv_nested(X, y, model, 'xgb - nested', space, 'yes')
res_xgb_nested.columns = ["y_true", "y_score"] 

### Shap Values XGBoost - nested

Here we compute the shap values for all features and safe the figure under "shap_all_features.pdf"

In [ ]:
X_df = pd.DataFrame(X, columns=audio)

explainer = shap.Explainer(model, X_df)
shap_values = explainer(X_df)

shap.plots.bar(shap_values, show=False)

ax = plt.gca()

for patch in ax.patches:
    patch.set_color("blue")

for text in ax.texts:
    text.set_color("blue")

plt.savefig("shap_all_features.pdf", bbox_inches="tight")
plt.close()

Here we compute the shap values for the top 5 features and safe teh figure under "shap_top5_features.pdf"

In [ ]:
X_df = pd.DataFrame(X, columns=audio) 

explainer = shap.Explainer(model, X_df)
shap_values = explainer(X_df)

mean_abs = np.abs(shap_values.values).mean(axis=0)      
top5_idx = np.argsort(mean_abs)[::-1][:5]               

shap_top5 = shap_values[:, top5_idx]

shap.plots.bar(shap_top5, max_display=5, show=False)


ax = plt.gca()
ax.tick_params(axis='y', labelsize=16)
ax.tick_params(axis='x', labelsize=14)
ax.xaxis.grid(True, linestyle="--", alpha=0.6)
ax.set_axisbelow(True)
ax.set_xlabel("Mean absolute SHAP value", fontsize=17)
color = plt.get_cmap("Blues")(0.85)
for p in ax.patches:
    p.set_facecolor(color)

for t in ax.texts:
    t.set_color(color)
    

plt.tight_layout()
plt.savefig("shap_top5_features.pdf", bbox_inches="tight")
plt.close()

### Shap swarm plot

Here we compute a SHAP swarm plot, rename the feature names
to more interpretable labels for visualization and save the plot under "shap_swarm_top5_features.pdf"

In [ ]:
feature_name_map = {
    "spectralFluxV_sma3nz_stddevNorm": "Variability of spectral flux",
    "spectrum_6": "Low-frequency spectral energy",
    "VoicedSegmentsPerSec": "Rate of voiced segments",
    "shimmerLocaldB_sma3nz_stddevNorm": "Variability of local shimmer",
    "spectrum_2": "Very-low-frequency spectral energy"
}

X_df = pd.DataFrame(X, columns=audio) 


explainer = shap.Explainer(model, X_df)
shap_values = explainer(X_df)

mean_abs = np.abs(shap_values.values).mean(axis=0)   
top5_idx = np.argsort(mean_abs)[::-1][:5]           

shap_top5 = shap_values[:, top5_idx]

top5_names = [X_df.columns[i] for i in top5_idx]

new_feature_names = [feature_name_map.get(name, name) for name in top5_names]

shap_top5.feature_names = new_feature_names

shap.plots.beeswarm(shap_top5, max_display=5, show=False)

fig = plt.gcf()
ax = plt.gca()
ax.tick_params(axis='y', labelsize=16)
ax.tick_params(axis='x', labelsize=14)
ax.xaxis.grid(True, linestyle="--", alpha=0.6)
ax.set_axisbelow(True)
ax.set_xlabel("SHAP value", fontsize=17)

for label in ax.get_yticklabels():
    label.set_color("black")

cbar_ax = fig.axes[-1]                     
cbar_ax.tick_params(labelsize=14)          
cbar_ax.set_ylabel("Feature value", fontsize=16)

for t in cbar_ax.texts:
    t.set_fontsize(14)
    
plt.tight_layout()
plt.savefig("shap_swarm_top5_features.pdf", bbox_inches="tight")
plt.close()

### All ROC Graph

Here we ese the previously stored prediction scores from different models
to plot their ROC curves in a single combined figure.

In [ ]:
predictions = {
    "LR": res_lr_nested["y_score"].values,

    "RF": res_rf_nested["y_score"].values,

    "SVM": res_svm_nested["y_score"].values,

    "XGBoost": res_xgb_nested["y_score"].values,
}

y_true = res_lr["y_true"].values

myml.all_roc_graph_colorblindness(y_true, predictions, name="all_models")